# Simple LoRA Adapter Test

Set a prompt and run one cell to see model output.

In [2]:
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ---- Edit these ----
BASE_MODEL = "Qwen/Qwen3.5-9B"
ADAPTER_PATH = "checkpoints/lora_nl_command_full"
MODE = "full"  # 'full' or 'tail'
SELECTED_TOOL = "cat"  # used only when MODE='tail'
# --------------------

project_root = Path.cwd()
if not (project_root / "checkpoints").exists() and (project_root.parent / "checkpoints").exists():
    project_root = project_root.parent

adapter_path = Path(ADAPTER_PATH)
if not adapter_path.is_absolute():
    adapter_path = project_root / adapter_path

print(f"Project root: {project_root}")
print(f"Adapter path: {adapter_path}")

if not adapter_path.exists():
    raise FileNotFoundError(f"Adapter path not found: {adapter_path}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {"trust_remote_code": True}
if torch.cuda.is_available():
    model_kwargs["torch_dtype"] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model_kwargs["device_map"] = "auto"
else:
    model_kwargs["torch_dtype"] = torch.float32

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
model = PeftModel.from_pretrained(base_model, str(adapter_path))
model.eval()

device = next(model.parameters()).device
print(f"Loaded on device: {device}")

/scratch4/home/akrik/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /scratch4/home/akrik/NTILC
Adapter path: /scratch4/home/akrik/NTILC/checkpoints/lora_nl_command_full


`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 51150.05it/s]
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 427/427 [00:02<00:00, 152.72it/s, Materializing param=model.norm.weight]                              


Loaded on device: cuda:1


In [3]:
def build_prompt(query: str, tool: str = "", mode: str = "full") -> str:
    if mode == "tail":
        return (
            "You map a user request to shell command arguments.\n"
            "Given the selected tool and request, output only the command tail (arguments and values).\n"
            "Do not repeat the tool name.\n"
            f"Tool: {tool}\n"
            f"User request: {query}\n"
            "Command tail:"
        )
    return (
        "You map a user request to exactly one Linux shell command.\n"
        "Output only the command and nothing else.\n"
        f"User request: {query}\n"
        "Command:"
    )

def generate(prompt_text: str, max_new_tokens: int = 96, temperature: float = 0.0, top_p: float = 1.0) -> str:
    prompt = build_prompt(prompt_text, tool=SELECTED_TOOL, mode=MODE)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-6),
            top_p=top_p,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids = out[0, inputs["input_ids"].shape[1]:]
    return prompt, tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

In [4]:
PROMPT = "create a directory named 'data' and move all .txt files into it"
prompt, result = generate(PROMPT)
print(f"Prompt:\n{prompt}\n")
print("Model output:")
print(result)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Prompt:
You map a user request to exactly one Linux shell command.
Output only the command and nothing else.
User request: create a directory named 'data' and move all .txt files into it
Command:

Model output:
mkdir -p data && mv *.txt data/
